# Full-run test for Copulagan

This notebook provides runnable cells to perform a full training run for `CopulaGAN` using the project `TrainTestSplitPipeline`.

Notes:
- The notebook defaults to skipping the pipeline TSTR evaluations (these import `xgboost`) to avoid extra dependency installs. Set `SKIP_EVALUATIONS = False` if you have `xgboost` installed and want full evaluation.
- Adjust the model config dictionaries below to control epochs / steps / batch sizes for real full runs.
- Each cell is annotated so you can run the cells interactively per dataset/model.

In [1]:
pip install copulas

In [2]:
pip install sdv

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Convenience wrapper to create a pipeline that optionally disables evaluations
def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [4]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [5]:
# User configuration: choose dataset(s) and run options
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']  # change or add: e.g., ['adult','car']
# If True, the pipeline will NOT run TSTR evaluations (avoids needing xgboost)
SKIP_EVALUATIONS = False

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

# Model mapping
MODEL_MAP = {
    'ctgan': ('katabatic.models.ctgan.models', 'CTGANModel'),
    'ctabgan': ('katabatic.models.ctabgan.models', 'CTABGANModel'),
    'tvae': ('katabatic.models.tvae.models', 'TVAEModel'),
    'copulagan': ('katabatic.models.copulagan.models', 'CopulaGANModel'),  # NEW
    'tabddpm': ('katabatic.models.tabddpm.models', 'Tabddpm'),
    'tabsyn': ('katabatic.models.tabsyn.models', 'TabSyn'),
}

# CopulaGAN Configuration (from CTGAN library defaults)
COPULAGAN_CONFIG = {
    'epochs': 300,
    'batch_size': 500,
    'generator_dim': (256, 256),
    'discriminator_dim': (256, 256),
    'generator_lr': 2e-4,
    'discriminator_lr': 2e-4,
    'discriminator_steps': 1,
    'pac': 10,
    'cuda': True,
}

CTABGAN_CONFIG = {
    'epochs': 150,
    'batch_size': 500,
    'class_dim': (256, 256, 256, 256),
    'random_dim': 100,
    'num_channels': 64,
    'l2scale': 1e-5,
    'test_ratio': 0.20,
}

# Default full-run configs for each model (tweak as needed)
CTGAN_CONFIG = {
    'epochs': 200,
    'batch_size': 512,
    'noise_dim': 128,
    'backend': 'torch',
}

TABDDPM_CONFIG = {
    # TabDDPM expects a 'config' passed into train via pipeline; pipeline.run will pass through kwargs to model.train
    'config': {
        'steps': 1000,
        'num_timesteps': 500,
        'batch_size': 64,
        'use_ema': True,
        'd_layers': (128,128),
    }
}

TABSYN_CONFIG = {
    'decoder_epochs': 50,
    'decoder_batch_size': 1024,
    'diffusion_epochs': 200,
    'diffusion_batch_size': 256,
}

## Preprocess datasets (run once)
Run this cell to discretize the raw CSVs into `discretized_data/{dataset}.csv`. 

In [6]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(file_path=f'raw_data/{dataset}.csv', output_path=f'discretized_data/{dataset}.csv', bins=10, strategy='uniform')
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        import traceback; traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run full Copulagan
This cell runs CTGAN for each selected dataset using `CTGAN_CONFIG` above. Be patient — full training can take time depending on `epochs` and dataset size.

In [7]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'CopulaGAN -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'copulagan')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['copulagan']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**COPULAGAN_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('CopulaGAN finished for', dataset, '-> result:', result)
    except Exception as e:
        print('CopulaGAN failed for', dataset, e)
        import traceback; traceback.print_exc()


CopulaGAN -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[CopulaGAN] Detected discrete columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'class']
[CopulaGAN] Creating metadata...
[CopulaGAN] Initializing CopulaGAN with 300 epochs...
[CopulaGAN] Training on 26048 samples...
[CopulaGAN] Finished training in 1296.92 seconds.
[CopulaGAN] Generating 26048 synthetic samples...
[CopulaGAN] Synthetic data saved:
  X -> synthetic\adult\copulagan\x_synth.csv
  y -> synthetic\adult\copulagan\y_synth.csv

Results saved to: Result